In [12]:
import torch
import torch.nn as nn
import pandas as pd
from transformers import BertModel, BertTokenizer
from torchvision import models, transforms
from torch.utils.data import DataLoader, Dataset
from PIL import Image
from tqdm.auto import tqdm
import os

LOCAL_BERT_PATH = "/mnt/workspace/FRD-CLIP/models/bert-base-chinese"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

class SemanticEnhancer(nn.Module):
    """语义增强模块：通过自注意力强化单模态内的特征相关性"""
    def __init__(self, d_model, nhead=8):
        super().__init__()
        self.self_attn = nn.MultiheadAttention(d_model, nhead, batch_first=True)
        self.norm = nn.LayerNorm(d_model)

    def forward(self, x):
        attn_out, _ = self.self_attn(x, x, x)
        return self.norm(x + attn_out)

class CrossModalCoAttention(nn.Module):
    """跨模态共注意模块：实现图文深度对齐"""
    def __init__(self, d_model, nhead=8):
        super().__init__()
        self.cm_attn = nn.MultiheadAttention(d_model, nhead, batch_first=True)
        self.norm = nn.LayerNorm(d_model)

    def forward(self, query, key_value):
        attn_out, _ = self.cm_attn(query, key_value, key_value)
        return self.norm(query + attn_out)

class SCCN_Model(nn.Module):
    def __init__(self, bert_path, d_model=768):
        super().__init__()
        # 1. 骨干网络
        self.bert = BertModel.from_pretrained(bert_path)

        # 冻结BERT
        for param in self.bert.parameters():
            param.requires_grad = False

        resnet = models.resnet50(pretrained=True)
        self.visual_backbone = nn.Sequential(*list(resnet.children())[:-2])

        # 冻结Resnet
        for param in self.visual_backbone.parameters():
            param.requires_grad = False

        self.img_proj = nn.Linear(2048, d_model)

        # 2. 语义增强层 (Semantic Enhancement)
        self.text_enhancer = SemanticEnhancer(d_model)
        self.img_enhancer = SemanticEnhancer(d_model)

        # 3. 跨模态共注意层 (Cross-modal Co-attention)
        self.text_aware_img = CrossModalCoAttention(d_model)
        self.img_aware_text = CrossModalCoAttention(d_model)

        # 4. 分类器
        self.classifier = nn.Sequential(
            nn.Linear(d_model * 2, 32), # 维度从 512 砍到 32，造成信息瓶颈
            nn.ReLU(),
            nn.Dropout(0.8),           # Dropout 从 0.3 提到 0.8，丢弃 80% 的特征
            nn.Linear(32, 2)
        )

    def forward(self, input_ids, attention_mask, pixel_values):
        # 文本与图像基础特征提取
        text_feats = self.bert(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state
        img_feats = self.visual_backbone(pixel_values).flatten(2).transpose(1, 2)
        img_feats = self.img_proj(img_feats)

        # 语义增强 (Self-Attention)
        text_enhanced = self.text_enhancer(text_feats)
        img_enhanced = self.img_enhancer(img_feats)

        # 跨模态交互 (Co-Attention)
        # 文本引导图片，图片引导文本
        # t_final = self.img_aware_text(text_enhanced, img_enhanced)
        # i_final = self.text_aware_img(img_enhanced, text_enhanced)

        t_final = text_enhanced # 完全不看图像
        i_final = img_enhanced  # 完全不看文本

        # 聚合与拼接 (Global Average Pooling)
        t_vec = torch.mean(t_final, dim=1)
        i_vec = torch.mean(i_final, dim=1)
        
        combined = torch.cat((t_vec, i_vec), dim=1)
        return self.classifier(combined)

In [13]:
# --- 2. 专用 Dataset ---
# 注意：ResNet 的预处理与 CLIP 不同，需使用标准的 ImageNet 归一化
class SimpleFusionDataset(Dataset):
    def __init__(self, df, bert_tokenizer, max_len=128):
        self.df = df
        self.tokenizer = bert_tokenizer
        self.max_len = max_len
        self.transform = transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        text = str(row["text_clean"]) if pd.notna(row["text_clean"]) else ""
        img_path = row["img_local_path"]
        
        # 文本处理
        enc = self.tokenizer(text, padding='max_length', truncation=True, 
                             max_length=self.max_len, return_tensors="pt")
        
        # 图像处理
        try:
            image = Image.open(img_path).convert("RGB")
            pixel_values = self.transform(image)
        except:
            pixel_values = torch.zeros((3, 224, 224))
            
        return {
            "input_ids": enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "pixel_values": pixel_values,
            "label": torch.tensor(int(row["label"]), dtype=torch.long)
        }

In [14]:
import copy
import pandas as pd
from torch.utils.data import DataLoader
from tqdm import tqdm
from sklearn.metrics import classification_report, accuracy_score

def train_sccn():
    # 数据加载
    tokenizer = BertTokenizer.from_pretrained(LOCAL_BERT_PATH)
    train_loader = DataLoader(SimpleFusionDataset(pd.read_csv('/mnt/workspace/FRD-CLIP/tmp/train_ready.csv'), tokenizer), batch_size=32, shuffle=True)
    val_loader = DataLoader(SimpleFusionDataset(pd.read_csv('/mnt/workspace/FRD-CLIP/tmp/val_ready.csv'), tokenizer), batch_size=32)
    test_loader = DataLoader(SimpleFusionDataset(pd.read_csv('/mnt/workspace/FRD-CLIP/tmp/test_ready.csv'), tokenizer), batch_size=32)

    model = SCCN_Model(LOCAL_BERT_PATH).to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4) #lr=1e-5
    criterion = nn.CrossEntropyLoss()
    
    best_val_acc = 0.0
    save_path = "best_sccn_model.pth"

    for epoch in range(5):
        model.train()
        total_loss = 0
        for batch in tqdm(train_loader, desc=f"Epoch {epoch+1}"):
            optimizer.zero_grad()
            outputs = model(batch["input_ids"].to(DEVICE), batch["attention_mask"].to(DEVICE), batch["pixel_values"].to(DEVICE))
            loss = criterion(outputs, batch["label"].to(DEVICE))
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        
        # 验证
        val_loss, val_acc, _, _ = evaluate_logic(model, val_loader, criterion, DEVICE)
        print(f"Epoch {epoch+1} Complete. Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}")

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save(model.state_dict(), save_path)
            print(f"--> Saved Best Model with Acc: {val_acc:.4f}")

    # 测试
    print("\n" + "="*20 + " Test Set Evaluation " + "="*20)
    model.load_state_dict(torch.load(save_path))
    evaluate_on_test_set(model, test_loader, criterion, DEVICE)

In [15]:
from sklearn.metrics import classification_report, accuracy_score

@torch.no_grad()
def evaluate_logic(model, dataloader, criterion, device):
    """用于训练过程中的轻量化验证"""
    model.eval()
    total_loss = 0.0
    all_labels = []
    all_preds = []
    
    for batch in dataloader:
        ids = batch["input_ids"].to(device)
        mask = batch["attention_mask"].to(device)
        imgs = batch["pixel_values"].to(device)
        labels = batch["label"].to(device)
        
        outputs = model(ids, mask, imgs)
        loss = criterion(outputs, labels)
        total_loss += loss.item()
        
        preds = torch.argmax(outputs, dim=1)
        all_labels.extend(labels.cpu().numpy())
        all_preds.extend(preds.cpu().numpy())
    
    acc = accuracy_score(all_labels, all_preds)
    return total_loss / len(dataloader), acc, all_labels, all_preds

@torch.no_grad()
def evaluate_on_test_set(model, test_loader, criterion, device):
    """用于最后输出详细的测试报告"""
    _, acc, labels, preds = evaluate_logic(model, test_loader, criterion, device)
    
    print(f"\nFinal Results on Test Set:")
    print(f"Test Acc: {acc:.6f}")
    print("-" * 60)
    # 输出 precision, recall, f1 等
    print(classification_report(labels, preds, target_names=['real', 'fake'], digits=6))

In [16]:
train_sccn()

/opt/conda/lib/python3.9/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/opt/conda/lib/python3.9/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
Epoch 1: 100%|██████████| 171/171 [00:33<00:00,  5.05it/s]


Epoch 1 Complete. Val Loss: 0.4190, Val Acc: 0.8193
--> Saved Best Model with Acc: 0.8193


Epoch 2: 100%|██████████| 171/171 [00:35<00:00,  4.86it/s]


Epoch 2 Complete. Val Loss: 0.2600, Val Acc: 0.8938
--> Saved Best Model with Acc: 0.8938


Epoch 3: 100%|██████████| 171/171 [00:34<00:00,  4.89it/s]


Epoch 3 Complete. Val Loss: 0.2351, Val Acc: 0.8938


Epoch 4: 100%|██████████| 171/171 [00:33<00:00,  5.14it/s]


Epoch 4 Complete. Val Loss: 0.2230, Val Acc: 0.8955
--> Saved Best Model with Acc: 0.8955


Epoch 5: 100%|██████████| 171/171 [00:34<00:00,  4.98it/s]


Epoch 5 Complete. Val Loss: 0.1974, Val Acc: 0.9101
--> Saved Best Model with Acc: 0.9101

==================== Test Set Evaluation ====================

Final Results on Test Set:
Test Acc: 0.913601
------------------------------------------------------------
              precision    recall  f1-score   support

        real   0.880403  0.971383  0.923658       629
        fake   0.962105  0.846296  0.900493       540

    accuracy                       0.913601      1169
   macro avg   0.921254  0.908840  0.912075      1169
weighted avg   0.918144  0.913601  0.912957      1169



In [6]:
test_model = SCCN_Model(LOCAL_BERT_PATH)
print(test_model)

/opt/conda/lib/python3.9/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/opt/conda/lib/python3.9/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


SCCN_Model(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(21128, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_af